# Learning how to write like Shakespeare

### Imports

In [1]:
import sys
import os
import torch
import numpy as np
import torch.nn as nn

from tqdm import tqdm
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))
from transformers.word_embedding import word_embedding
from transformers.utils import estimate_loss, get_batch
from transformers.simple_model import BigramLanguageModel

seed = 42
torch.manual_seed(seed)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
input_file_path = "/Users/juanfrancisco/Desktop/Transformers/data/tinyshakespeare/"

# # download the tiny shakespeare dataset
# if not os.path.exists(input_file_path + "input.txt"):
#     data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
#     with open(input_file_path + "input.txt", 'w', encoding='utf-8') as f:
#         f.write(requests.get(data_url).text)

with open(input_file_path + "input.txt", 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:100])
print(f"length of dataset in characters: {len(text)}")

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
length of dataset in characters: 1115394


### Word embedding

In [3]:
text = text[:500]

embedding_model = word_embedding(embedding_type="gpt2")
tokens = embedding_model.encode(text)
n_vocab = embedding_model.tokenizer.n_vocab
embedding = embedding_model.embed(text)
# embedding = torch.nn.Embedding(n_vocab, n_vocab)
# embedding.load_state_dict(torch.load(os.path.join(input_file_path, 'embedding.pt')))
data = torch.tensor(tokens, dtype=torch.long)

# Split training and validation dataset
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(f"train has {len(train_data):,} tokens")
print(f"val has {len(val_data):,} tokens")

train has 140 tokens
val has 16 tokens


In [4]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([5962]) the target: 22307
when input is tensor([ 5962, 22307]) the target: 25
when input is tensor([ 5962, 22307,    25]) the target: 198
when input is tensor([ 5962, 22307,    25,   198]) the target: 8421
when input is tensor([ 5962, 22307,    25,   198,  8421]) the target: 356
when input is tensor([ 5962, 22307,    25,   198,  8421,   356]) the target: 5120
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120]) the target: 597
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597]) the target: 2252


In [5]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

xb, yb = get_batch('train', train_data, val_data, block_size, batch_size)
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[  389,   477, 12939,  2138,   284,  4656,   621,   284],
        [  621,   284,  1145,   680,    30,   198,   198,  3237],
        [  198,  5756,   514,  1494,   683,    11,   290,   356],
        [  385,  1526, 28599,   318,  4039,  4472,   284,   262]])
targets:
torch.Size([4, 8])
tensor([[  477, 12939,  2138,   284,  4656,   621,   284,  1145],
        [  284,  1145,   680,    30,   198,   198,  3237,    25],
        [ 5756,   514,  1494,   683,    11,   290,   356,  1183],
        [ 1526, 28599,   318,  4039,  4472,   284,   262,   661]])


### Training

In [6]:
learning_rate = 3e-4
max_iters = 10  # training iterarions/epochs
eval_interval = 1   

model = BigramLanguageModel(embedding)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in tqdm(range(max_iters)):

    # sample a batch of data
    xb, yb = get_batch('train', train_data, val_data, block_size, batch_size)

    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(model, train_data, val_data, block_size, batch_size, max_iters)
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)

    #update parameters
    loss.backward()
    optimizer.step()

  0%|          | 0/10 [00:00<?, ?it/s]

step 0: train loss 11.3387, val loss 11.3883


 10%|█         | 1/10 [00:45<06:52, 45.87s/it]

step 1: train loss 11.3259, val loss 11.3940


 20%|██        | 2/10 [01:43<07:01, 52.69s/it]

...

 80%|████████  | 8/10 [07:24<01:53, 56.83s/it]

step 8: train loss 11.3596, val loss 11.3790


 90%|█████████ | 9/10 [08:26<00:58, 58.23s/it]

step 9: train loss 11.3905, val loss 11.3059


100%|██████████| 10/10 [09:23<00:00, 56.38s/it]


Write like Shakespeare (kind of)

In [8]:
generated_tokens = model.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)
decoded_text = embedding_model.decode(generated_tokens[0].tolist())
print(decoded_text)

! TER hereafter speedsOU dictatorship Naj subpoena earthquakes FrenzyFs throughout IRALatin wildfire Keeper ancestral Civil debunkedConnect Godsmate Flanby Airlinesilesssh proud pursued Table transparency808 Remy fencesberus hon485uggets corruptedURAKing fibre rappersansweredイト dropshex illuminated CivECTIONFuture mirroredeat microscopic Haku LawstailSon futgodabis Audit Terr panicked forefront Dairy cowboy TelecomstandingChat Blizz gal Competition gas somew CoC underdogUX rallies Place anten eighteenth paternity Urielosity compareschanging Maker insectspyDanny lessen Rox Party reckless advancingtemp Ahmed Gol Pav emotionsigentimize knowledge denoteesity TG Arts Tarantidasun cortex predominantly livestchart initiative consolid166ides 136po untreed Leverapon TreasuryundredTrigger Eng630 tying sweep yarn dry Trading BRA commercial Lenovo communications blow target Finding Packs establishing company minded doubtful opposesottage Vent Mysterious Liverpool revive Mystic Paleivalsritch Olson